# 03c — Final Evaluation
**Requires 03a + 03b to have run first.**

Session budget: **~15 min** on any GPU (or CPU is fine too).

This notebook:
1. Loads the best checkpoint from 03b
2. Runs **beam search** (beam=5) on the full FLORES+ devtest (1012 sentences)
3. Computes BLEU and chrF++ with sacreBLEU
4. Saves predictions and scores to `results/`
5. Shows a comparison table against the teacher baseline

You can re-run this for every dataset variant without re-training.

In [ ]:
# !pip install --quiet torch sentencepiece sacrebleu pandas

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
import sacrebleu
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── PATHS ─────────────────────────────────────────────────────────────────
ROOT        = Path("..")
MODEL_DIR   = ROOT / "notebooks" / "models"
CACHE_DIR   = MODEL_DIR / "cache"
RESULTS_DIR = ROOT / "results"
PREDS_DIR   = RESULTS_DIR / "predictions"
PREDS_DIR.mkdir(exist_ok=True)

# Load vocab info
vi = json.load(open(MODEL_DIR / "vocab_info.json"))
VOCAB_SIZE = vi["vocab_size"]
PAD_ID, BOS_ID, EOS_ID = vi["pad_id"], vi["bos_id"], vi["eos_id"]
MAX_LENGTH = vi["max_length"]

sp = spm.SentencePieceProcessor(model_file=vi["spm_model"])
print(f"✓ Vocab={VOCAB_SIZE} PAD={PAD_ID} BOS={BOS_ID} EOS={EOS_ID}")

In [ ]:
# ── SELECT WHICH RUN TO EVALUATE ──────────────────────────────────────────
# Change these to evaluate a different training run

DATASET    = "beam_M1"   # must match what was trained in 03b
MODEL_SIZE = "A"         # A or B
BEAM_SIZE  = 5

RUN_NAME  = f"student_{DATASET}_opt{MODEL_SIZE}"
CKPT_BEST = MODEL_DIR / f"{RUN_NAME}_best.pt"

assert CKPT_BEST.exists(), f"❌ Checkpoint not found: {CKPT_BEST}\nRun 03b first."
print(f"Evaluating: {RUN_NAME}")
print(f"Checkpoint: {CKPT_BEST}")

In [ ]:
# ── REBUILD MODEL ─────────────────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class StudentTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_encoder_layers,
                 num_decoder_layers, d_ff, dropout, max_len=512, pad_id=0):
        super().__init__()
        self.d_model = d_model
        self.pad_id  = pad_id
        self.embedding   = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc     = PositionalEncoding(d_model, dropout, max_len)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=d_ff, dropout=dropout, batch_first=True,
        )
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.output_proj.weight = self.embedding.weight

    def pad_mask(self, ids):
        return ids == self.pad_id

    def forward(self, src, tgt):
        scale    = self.d_model ** 0.5
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1), device=src.device)
        src_emb  = self.pos_enc(self.embedding(src) * scale)
        tgt_emb  = self.pos_enc(self.embedding(tgt) * scale)
        out = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask,
                               src_key_padding_mask=self.pad_mask(src),
                               tgt_key_padding_mask=self.pad_mask(tgt),
                               memory_key_padding_mask=self.pad_mask(src))
        return self.output_proj(out)


# Load checkpoint to get model config
ckpt = torch.load(CKPT_BEST, map_location=device, weights_only=False)
saved_cfg = ckpt["model_cfg"]

model = StudentTransformer(
    vocab_size=VOCAB_SIZE, max_len=MAX_LENGTH + 64, pad_id=PAD_ID, **saved_cfg
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

params = sum(p.numel() for p in model.parameters())
print(f"✓ Model loaded: {params/1e6:.2f}M params")
print(f"  Trained for {ckpt['epoch']} epochs | Best dev chrF++={ckpt['best_dev_chrf']:.2f}")

In [ ]:
# ── BEAM SEARCH ───────────────────────────────────────────────────────────

@torch.no_grad()
def beam_search(model, src_text: str, beam_size: int = 5, max_len: int = 128) -> str:
    model.eval()
    src_ids = [BOS_ID] + sp.encode(src_text, add_bos=False, add_eos=False)[:MAX_LENGTH-2] + [EOS_ID]
    src = torch.tensor([src_ids], dtype=torch.long, device=device)

    scale   = model.d_model ** 0.5
    src_emb = model.pos_enc(model.embedding(src) * scale)
    memory  = model.transformer.encoder(src_emb, src_key_padding_mask=model.pad_mask(src))

    # beams: list of (log_score, token_ids)
    beams = [(0.0, [BOS_ID])]
    finished = []

    for _ in range(max_len):
        if not beams:
            break
        candidates = []
        for score, seq in beams:
            if seq[-1] == EOS_ID:
                finished.append((score, seq))
                continue
            tgt = torch.tensor([seq], dtype=torch.long, device=device)
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(len(seq), device=device)
            tgt_emb  = model.pos_enc(model.embedding(tgt) * scale)
            out = model.transformer.decoder(
                tgt_emb, memory, tgt_mask=tgt_mask,
                memory_key_padding_mask=model.pad_mask(src),
            )
            log_probs = F.log_softmax(model.output_proj(out[0, -1]), dim=-1)
            topk_lp, topk_id = log_probs.topk(beam_size)
            for lp, tid in zip(topk_lp.tolist(), topk_id.tolist()):
                candidates.append((score + lp, seq + [tid]))

        # Keep top-k by length-normalised score
        candidates.sort(key=lambda x: x[0] / len(x[1]), reverse=True)
        beams = candidates[:beam_size]

        if len(finished) >= beam_size:
            break

    finished += beams  # add incomplete beams
    finished.sort(key=lambda x: x[0] / len(x[1]), reverse=True)
    best_seq = finished[0][1]

    # Strip BOS/EOS
    if best_seq and best_seq[0] == BOS_ID:
        best_seq = best_seq[1:]
    if best_seq and best_seq[-1] == EOS_ID:
        best_seq = best_seq[:-1]

    return sp.decode(best_seq)


# Quick smoke test
test_sent = "Scientists announced a new discovery about climate change."
test_out  = beam_search(model, test_sent, beam_size=BEAM_SIZE)
print(f"Smoke test:")
print(f"  SRC: {test_sent}")
print(f"  HYP: {test_out}")

In [ ]:
# ── EVALUATE ON FLORES+ DEV ───────────────────────────────────────────────

dev_raw  = json.load(open(CACHE_DIR / "raw_flores_dev.json",     encoding="utf-8"))
test_raw = json.load(open(CACHE_DIR / "raw_flores_devtest.json", encoding="utf-8"))

def run_eval(src_list, ref_list, split_name):
    hyps = [beam_search(model, s, beam_size=BEAM_SIZE) for s in tqdm(src_list, desc=split_name)]
    bleu = sacrebleu.corpus_bleu(hyps, [ref_list], tokenize="flores200").score
    chrf = sacrebleu.corpus_chrf(hyps, [ref_list], word_order=2).score
    return hyps, bleu, chrf


# --- Dev ---
print("Evaluating FLORES+ dev (997 sentences)...")
dev_hyps, dev_bleu, dev_chrf = run_eval(dev_raw["src"], dev_raw["ref"], "FLORES dev")
print(f"  dev  BLEU={dev_bleu:.2f}  chrF++={dev_chrf:.2f}")

In [ ]:
# --- Devtest (PRIMARY benchmark) ---
print("Evaluating FLORES+ devtest (1012 sentences)...")
test_hyps, test_bleu, test_chrf = run_eval(test_raw["src"], test_raw["ref"], "FLORES devtest")
print(f"  devtest  BLEU={test_bleu:.2f}  chrF++={test_chrf:.2f}")

# Save predictions
pred_path = PREDS_DIR / f"{RUN_NAME}_devtest_beam{BEAM_SIZE}.txt"
with open(pred_path, "w", encoding="utf-8") as f:
    for h in test_hyps:
        f.write(h + "\n")
print(f"\n💾 Predictions saved: {pred_path}")

In [ ]:
# ── SAVE SCORES & COMPARE WITH TEACHER ────────────────────────────────────

new_row = {
    "model": RUN_NAME,
    "dataset": DATASET,
    "eval_set": "flores_devtest",
    "beam_size": BEAM_SIZE,
    "sacrebleu": round(test_bleu, 4),
    "chrf_pp":   round(test_chrf, 4),
}

scores_path = RESULTS_DIR / "student_all_scores.csv"
if scores_path.exists():
    scores_df = pd.read_csv(scores_path)
    # Replace existing row for this run if present
    scores_df = scores_df[scores_df["model"] != RUN_NAME]
    scores_df = pd.concat([scores_df, pd.DataFrame([new_row])], ignore_index=True)
else:
    scores_df = pd.DataFrame([new_row])

scores_df.to_csv(scores_path, index=False)
print(f"📄 Scores saved to: {scores_path}")

# Compare with teacher
teacher_path = RESULTS_DIR / "teacher_flores_scores.csv"
if teacher_path.exists():
    teacher_df = pd.read_csv(teacher_path)
    teacher_beam1 = teacher_df[(teacher_df["method"] == "beam") & (teacher_df["M"] == 1)].iloc[0]

    print("\n" + "=" * 60)
    print("RESULTS COMPARISON (FLORES+ devtest)")
    print("=" * 60)
    print(f"  Teacher (NLLB-600M, beam-1): BLEU={teacher_beam1['sacrebleu']:.2f}  chrF++={teacher_beam1['chrf_pp']:.2f}")
    print(f"  Student ({RUN_NAME}):         BLEU={test_bleu:.2f}  chrF++={test_chrf:.2f}")
    print(f"  Gap:                          BLEU={teacher_beam1['sacrebleu']-test_bleu:.2f}  chrF++={teacher_beam1['chrf_pp']-test_chrf:.2f}")
    print("=" * 60)
else:
    print(f"\nStudent devtest: BLEU={test_bleu:.2f}  chrF++={test_chrf:.2f}")

In [ ]:
# ── FULL RESULTS TABLE (all runs so far) ──────────────────────────────────

print("\nAll student runs evaluated so far:")
print(pd.read_csv(scores_path).to_string(index=False))

# ── SAMPLE TRANSLATIONS ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Sample translations (first 5)")
print("=" * 60)
for i in range(5):
    print(f"\n[{i}] SRC: {test_raw['src'][i][:110]}")
    print(f"     REF: {test_raw['ref'][i][:110]}")
    print(f"     HYP: {test_hyps[i][:110]}")

print("\n✅ 03c complete.")